# Ramo EPOC X: pré-processamento e features

Este notebook processa, com o mesmo código, o ds004504 adaptado ao EPOC X e os registros do próprio
EPOC X. Ele não altera nada do pipeline original (`preprocessingData.ipynb`, na raiz): a
configuração fica em `config_epocx.py` e todas as saídas ficam nesta pasta.

| Etapa | O que faz |
|---|---|
| Canais e taxa | só os 10 canais comuns ao ds004504 e ao EPOC X, a 128 Hz |
| Duração | gravações com menos de 120 s são descartadas |
| Canais defeituosos | variância muito baixa ou acima de 3x a mediana; mais de 4 descarta o sujeito, senão interpola |
| Filtros | FIR 0,1-45 Hz e notch na rede elétrica da base (50 Hz na Grécia, 60 Hz no Brasil) |
| Referência | região das orelhas/mastoides (a de gravação nas duas bases) |
| Limpeza | ICA picard + wICA suave 1.5 seletivo; canais interpolados reconstruídos do sinal limpo |
| Épocas | 4 s a cada 2 s, com AutoReject |
| Features | as 31 medidas de `ds004504_features.py` (raiz) |

| Caminho (em `epocx/`) | Conteúdo |
|---|---|
| `dados/` | os `.edf` exportados do EmotivPRO (não versionados) |
| `features/ds004504/<identificador>/`, `features/epocx/<identificador>/` | cache: CSV de features e JSON de controle de qualidade por gravação |
| `eeg_features_ds004504.csv`, `eeg_features_epocx.csv` | features consolidadas (com `.meta.json`) |
| `qc_ds004504.csv`, `qc_epocx.csv` | controle de qualidade por gravação |

In [1]:
import sys, importlib.util, importlib.metadata as md
from pathlib import Path

print("Python do kernel:", sys.executable)
spec = importlib.util.find_spec("matplotlib")
PASTA_MPL = Path(spec.origin).parent if spec else None
print("matplotlib em:", PASTA_MPL)
if PASTA_MPL:
    print("registry.py existe?", (PASTA_MPL / "backends" / "registry.py").exists())
try:
    print("versão registrada pelo pip:", md.version("matplotlib"))
except md.PackageNotFoundError:
    print("o pip não encontra o matplotlib")

Python do kernel: c:\Users\isabe\AppData\Local\Programs\Python\Python314\python.exe
matplotlib em: c:\Users\isabe\AppData\Local\Programs\Python\Python314\Lib\site-packages\matplotlib
registry.py existe? True
versão registrada pelo pip: 3.11.2


In [2]:
import subprocess, shutil

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "matplotlib"])

site = PASTA_MPL.parent
for p in [PASTA_MPL, *site.glob("matplotlib-*.dist-info")]:
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
        print("apagado:", p)

r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "matplotlib"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
print(r.stderr[-1500:])
print("registry.py existe agora?", (PASTA_MPL / "backends" / "registry.py").exists())

ement already satisfied: numpy>=1.25 in c:\Users\isabe\AppData\Local\Programs\Python\Python314\Lib\site-packages (from matplotlib) (2.5.3)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 3.2 MB/s eta 0:00:03
   --------------- ------------------------ 3.7/9.5 MB 11.6 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.5 MB 13.8 MB/s eta 0:00:01
   ----------------------------------- ---- 8.4/9.5 MB 11.5 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 12.0 MB/s  0:00:00


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

registry.py existe agora? True


In [3]:
import matplotlib, matplotlib.backends.registry
print(matplotlib.__version__)

3.11.2


In [4]:
import sys, json
from pathlib import Path
import pandas as pd

# raiz do projeto = a pasta que contém feature_sets.py (sobe a partir da pasta atual)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "feature_sets.py").exists())
EPOCX_ROOT = PROJECT_ROOT / "epocx"
for p in (str(PROJECT_ROOT), str(EPOCX_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

import config_epocx as CFG
import preprocessing as P
import wica

DATA_ROOT = PROJECT_ROOT / "88 subjects"
DATA_DIR = EPOCX_ROOT / "dados"
TAG = CFG.feature_tag()
CACHE_DS = EPOCX_ROOT / "features" / "ds004504" / TAG
CACHE_EPOCX = EPOCX_ROOT / "features" / "epocx" / TAG

def save_features(frames, name, extra=None):
    df = pd.concat(frames, ignore_index=True)
    out = EPOCX_ROOT / name
    df.to_csv(out, index=False)
    out.with_suffix(".meta.json").write_text(json.dumps({
        "feature_tag": TAG, "channels": CFG.CHANNELS, "sfreq": CFG.TARGET_SFREQ,
        "reference": CFG.TARGET_REFERENCE, "n_recordings": int(df["Subject_ID"].nunique()),
        "n_epochs": int(len(df)), **(extra or {})}, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{out.relative_to(PROJECT_ROOT)}: {df['Subject_ID'].nunique()} gravações, {len(df)} épocas")
    return df

print("identificador:", TAG)
print(f"canais ({len(CFG.CHANNELS)}):", CFG.CHANNELS, "| taxa:", CFG.TARGET_SFREQ, "Hz")
print("referência-alvo:", CFG.TARGET_REFERENCE)
for nome, cfg in P.DATASETS.items():
    print(f"  {nome}: rede {cfg['line_freq']:.0f} Hz, referência de gravação '{cfg['source_reference']}'")
print(f"wICA: {wica.WAVELET}, limiar x{wica.THRESHOLD_MULT}, suave={wica.SOFT}, curtose mínima {wica.KURTOSIS_MIN}")
print(f"duração mínima {P.MIN_DURATION_S:.0f} s | máximo de canais defeituosos {P.MAX_NOISY}")

identificador: v3.0-mastoid-epocx_comum-128Hz
canais (10): ['F7', 'F3', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'F4', 'F8'] | taxa: 128.0 Hz
referência-alvo: mastoid
  ds004504: rede 50 Hz, referência de gravação 'linked_ears'
  epocx: rede 60 Hz, referência de gravação 'cms_mastoid'
wICA: coif5, limiar x1.5, suave=True, curtose mínima 8.0
duração mínima 120 s | máximo de canais defeituosos 4


## 1. ds004504 adaptado ao EPOC X

Seleção pelo `participants.tsv` (grupos A e C; acrescente "F" em `GRUPOS` para incluir a demência
frontotemporal). Os arquivos são lidos da pasta `88 subjects` na raiz do projeto.

In [5]:
GRUPOS = ["A", "C"]

participants = pd.read_csv(DATA_ROOT / "participants.tsv", sep="\t").rename(columns={"participant_id": "Subject_ID"})
amostra = participants[participants["Group"].isin(GRUPOS)].copy()
amostra["path"] = [DATA_ROOT / s / "eeg" / f"{s}_task-eyesclosed_eeg.set" for s in amostra["Subject_ID"]]
amostra = amostra[amostra["path"].map(Path.exists)]
print(amostra.groupby("Group").size().to_string())

# conferência de um sujeito antes do lote
r0 = amostra.iloc[0]
P.process_subject(r0["path"], "ds004504", CACHE_DS, overwrite=True)
qc0 = json.loads((CACHE_DS / f"{r0['Subject_ID']}_qc.json").read_text(encoding="utf-8"))
print(json.dumps({k: v for k, v in qc0["preprocessing"].items() if k != "wica_ics"}, indent=1, ensure_ascii=False))
pd.DataFrame(qc0["preprocessing"].get("wica_ics", []))

Group
A    36
C    29


c:\Users\isabe\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dropped 5 epochs: 63, 188, 223, 224, 225
  canais defeituosos ['O1'] | ICA 9 comp. | wICA tratou 3/9 ICs, 3.4% da variância em média | 293/298 épocas mantidas
{
 "dataset": "ds004504",
 "channels": [
  "F7",
  "F3",
  "T7",
  "P7",
  "O1",
  "O2",
  "P8",
  "T8",
  "F4",
  "F8"
 ],
 "sfreq": 128.0,
 "duration_s": 599.8,
 "noisy_channels": [
  "O1"
 ],
 "notch_hz": [
  50.0
 ],
 "reference": {
  "target": "mastoid",
  "source": "linked_ears",
  "ref_channels": [],
  "n_channels_after": 9,
  "detected_before": {
   "n_eeg": 10,
   "avg_ratio": 1.1475,
   "looks_average": false,
   "ear_or_mastoid_channels": [],
   "custom_ref_applied": false
  }
 },
 "ica_n_components": 9,
 "ica_n_theory": 9,
 "ica_rank": 9,
 "wavelet": "coif5",
 "wica_level": 8,
 "wica_threshold_mult": 1.5,
 "wica_soft": true,
 "wica_kurtosis_min": 8.0,
 "wica_n_ics": 9,
 "wica_n_ics_treated": 3,
 "wica_var_removed_mean_pct": 3.3654137559096258,
 "wica_var_removed_max_pct": 14.959746713206231,
 "wica_projection_error": 

,ic,curtose,tratada,var_removida_pct
0,0,0.6009,False,0.0000
1,1,1.9478,False,0.0000
2,2,5.1670,False,0.0000
3,3,0.6537,False,0.0000
4,4,7.2872,False,0.0000
5,5,1.2040,False,0.0000
6,6,17.9221,True,14.9597
7,7,18.7193,True,12.2819
8,8,9.7299,True,3.0471


In [6]:
erros_ds = P.run(list(amostra["path"]), "ds004504", CACHE_DS)
print("\nerros:", erros_ds or "nenhum")

frames = [pd.read_csv(f) for f in sorted(CACHE_DS.glob("*_epochs.csv"))]
cov = [c for c in ("Group", "Age", "Gender", "MMSE") if c in amostra.columns]
frames = [f.merge(amostra[["Subject_ID"] + cov], on="Subject_ID", how="inner") for f in frames]
epochs_ds = save_features(frames, "eeg_features_ds004504.csv")
qc_ds = P.qc_table(CACHE_DS)
qc_ds.to_csv(EPOCX_ROOT / "qc_ds004504.csv", index=False)
print(qc_ds["status"].value_counts().to_string())

[1/65] sub-001
[2/65] sub-002
Dropped 37 epochs: 21, 22, 55, 57, 61, 62, 102, 104, 127, 128, 130, 136, 153, 154, 164, 174, 178, 185, 186, 188, 208, 209, 242, 250, 251, 256, 257, 261, 277, 281, 282, 284, 304, 355, 356, 364, 368
  canais defeituosos nenhum | ICA 10 comp. | wICA tratou 4/10 ICs, 5.6% da variância em média | 358/395 épocas mantidas
[3/65] sub-003
Dropped 1 epoch: 118
  canais defeituosos nenhum | ICA 10 comp. | wICA tratou 0/10 ICs, 0.0% da variância em média | 151/152 épocas mantidas
[4/65] sub-004
Dropped 19 epochs: 14, 68, 86, 102, 140, 143, 178, 188, 206, 211, 223, 239, 240, 269, 285, 286, 334, 335, 345
  canais defeituosos ['O1', 'O2'] | ICA 8 comp. | wICA tratou 1/8 ICs, 0.7% da variância em média | 333/352 épocas mantidas
[5/65] sub-005
Dropped 4 epochs: 0, 309, 375, 376
  canais defeituosos ['F3', 'F4'] | ICA 8 comp. | wICA tratou 7/8 ICs, 12.4% da variância em média | 397/401 épocas mantidas
[6/65] sub-006
Dropped 2 epochs: 219, 243
  canais defeituosos nenhum | I

## 2. EPOC X

Os dados estão em formato BIDS: `epocx/dados/sub-XXX/ses-XX/eeg/`. Só as gravações de **olhos
fechados** entram, como no ds004504; quem tem duas (run-1 e run-2) contribui com as duas, e a
comparação usa a média por participante.

De cada gravação, o código lê também o `_eeg.json` (a frequência da rede, quando informada,
substitui o padrão de 60 Hz; a referência declarada fica registrada) e o `_channels.tsv` (canais
marcados como ruins entram como defeituosos, além dos detectados pela variância). Se houver um
`participants.tsv` em `epocx/dados/`, idade e sexo são acrescentados às features.

In [9]:
import importlib
P = importlib.reload(P)

In [10]:
TAREFA = "eyesclosed"          # use None para incluir todas as tarefas

gravacoes = P.discover_bids(DATA_DIR, task=TAREFA)
assert len(gravacoes), f"nenhuma gravação '{TAREFA}' encontrada em {DATA_DIR}"
print(f"{len(gravacoes)} gravações de {gravacoes['Subject_ID'].nunique()} participantes")
print(gravacoes.groupby("Subject_ID").size().value_counts().rename("participantes com N gravações").to_string())
print("sem _eeg.json:", gravacoes.loc[~gravacoes["has_eeg_json"], "Recording_ID"].tolist() or "nenhuma")

participantes_epocx = P.read_participants(DATA_DIR)
print("participants.tsv:", "encontrado" if participantes_epocx is not None else "não encontrado")

# conferência de uma gravação antes do lote
g0 = gravacoes.iloc[0]
P.process_subject(g0["path"], "epocx", CACHE_EPOCX, overwrite=True)
qc_e0 = json.loads((CACHE_EPOCX / f"{g0['Recording_ID']}_qc.json").read_text(encoding="utf-8"))
print("metadados BIDS:", json.dumps(qc_e0["bids"], indent=1, ensure_ascii=False))
print("rede usada:", qc_e0["line_freq_used"], "Hz")
print(json.dumps({k: v for k, v in qc_e0["preprocessing"].items() if k != "wica_ics"}, indent=1, ensure_ascii=False))

28 gravações de 14 participantes
2    14
sem _eeg.json: nenhuma
participants.tsv: encontrado
Dropped 12 epochs: 0, 1, 41, 42, 43, 52, 53, 66, 67, 68, 80, 81
  canais defeituosos ['P7'] | ICA 9 comp. | wICA tratou 2/9 ICs, 5.8% da variância em média | 77/89 épocas mantidas
metadados BIDS: {
 "eeg_json": {
  "SamplingFrequency": 128.0,
  "PowerLineFrequency": 60.0,
  "EEGReference": "CMS at left mastoid",
  "EEGGround": "DRL at right mastoid",
  "SoftwareFilters": "n/a",
  "HardwareFilters": {
   "Bandpass": {
    "HighpassCutoff (Hz)": 0.16,
    "LowpassCutoff (Hz)": 43
   },
   "Sinc filter": {
    "Order": 5,
    "Type": "digital, built-in"
   },
   "Notch": {
    "CenterFrequencies (Hz)": [
     50,
     60
    ]
   }
  },
  "RecordingDuration": 180.9921875,
  "Manufacturer": "Emotiv",
  "ManufacturersModelName": "EPOC X",
  "TaskName": "eyesclosed",
  "EEGChannelCount": 14
 },
 "bad_channels": []
}
rede usada: 60.0 Hz
{
 "dataset": "epocx",
 "channels": [
  "F7",
  "F3",
  "T7",
  "

In [11]:
erros_epocx = P.run(list(gravacoes["path"]), "epocx", CACHE_EPOCX)
print("\nerros:", erros_epocx or "nenhum")

frames = [pd.read_csv(CACHE_EPOCX / f"{r}_epochs.csv") for r in gravacoes["Recording_ID"]
          if (CACHE_EPOCX / f"{r}_epochs.csv").exists()]
if frames and participantes_epocx is not None:
    frames = [f.merge(participantes_epocx, on="Subject_ID", how="left") for f in frames]
if frames:
    epochs_epocx = save_features(frames, "eeg_features_epocx.csv", extra={"task": TAREFA})
    print(f"  {epochs_epocx['Subject_ID'].nunique()} participantes")
qc_epocx = P.qc_table(CACHE_EPOCX)
qc_epocx = qc_epocx[qc_epocx["Recording_ID"].isin(gravacoes["Recording_ID"])]
qc_epocx.to_csv(EPOCX_ROOT / "qc_epocx.csv", index=False)
print(qc_epocx["status"].value_counts().to_string())
print("referência declarada nos _eeg.json:", qc_epocx["eeg_reference_bids"].dropna().unique().tolist() or "não informada")
print("frequência da rede usada:", qc_epocx["line_freq_used"].dropna().unique().tolist())

[1/28] sub-012_ses-01_task-eyesclosed_run-1
[2/28] sub-012_ses-01_task-eyesclosed_run-2
Dropped 7 epochs: 1, 2, 3, 15, 20, 78, 79
  canais defeituosos nenhum | ICA 10 comp. | wICA tratou 1/10 ICs, 0.2% da variância em média | 82/89 épocas mantidas
[3/28] sub-022_ses-01_task-eyesclosed_run-1
  canais defeituosos ['F3'] | ICA 9 comp. | wICA tratou 6/9 ICs, 21.1% da variância em média | 94/94 épocas mantidas
[4/28] sub-022_ses-01_task-eyesclosed_run-2
Dropped 8 epochs: 22, 23, 57, 58, 73, 74, 83, 84
  canais defeituosos ['F4'] | ICA 9 comp. | wICA tratou 3/9 ICs, 1.4% da variância em média | 82/90 épocas mantidas
[5/28] sub-071_ses-01_task-eyesclosed_run-1
Dropped 60 epochs: 0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 22, 23, 25, 26, 27, 28, 29, 30, 33, 34, 35, 36, 37, 38, 39, 47, 48, 49, 50, 51, 52, 53, 54, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 84, 85, 86, 87, 88, 89, 90, 91
  canais defeituosos ['P7'] | ICA 9 comp. | wICA tratou 5/9 ICs, 6.6% da variância em méd

## 3. Controle de qualidade das duas bases

Diferenças grandes entre as bases (canais defeituosos, componentes tratadas pelo wICA, variância
removida, épocas rejeitadas) indicam que a limpeza não age igual nas duas, o que também vira
diferença nas features.

In [12]:
if "qc_ds" not in globals():
    qc_ds = pd.read_csv(EPOCX_ROOT / "qc_ds004504.csv")

qc_all = pd.concat([qc_ds.assign(base="ds004504"), qc_epocx.assign(base="EPOC X")], ignore_index=True)
ok = qc_all[qc_all["status"] == "ok"]
n_def = ok["noisy_channels"].fillna("").astype(str).map(lambda s: len([c for c in s.split(",") if c]))
resumo = ok.assign(n_defeituosos=n_def).groupby("base").agg(
    gravacoes=("Subject_ID", "count"),
    canais_defeituosos_med=("n_defeituosos", "median"),
    ics_tratadas_med=("wica_n_ics_treated", "median"),
    var_removida_med_pct=("wica_var_removed_mean_pct", "median"),
    epocas_med=("n_epochs", "median"),
    rejeitadas_med_pct=("pct_epochs_rejected", "median"),
)
print("descartados:", qc_all.loc[qc_all["status"] != "ok", ["base", "Subject_ID", "status"]].values.tolist() or "nenhum")
resumo.round(2)

descartados: nenhum


,gravacoes,canais_defeituosos_med,ics_tratadas_med,var_removida_med_pct,epocas_med,rejeitadas_med_pct
base,,,,,,
EPOC X,28,1.0,4.0,8.34,83.5,9.25
ds004504,65,1.0,4.0,4.77,379.0,3.50
